# ArtExtract – Task 2: Painting Similarity Search
### GSoC 2025 Evaluation Test | HumanAI / CERN

**Dataset:** [National Gallery of Art Open Data](https://github.com/NationalGalleryOfArt/opendata)  
**Goal:** Build a model to find similarities between paintings (faces, poses, style, composition).

---
## Table of Contents
1. [Strategy & Approach](#strategy)
2. [Dataset Loading & Exploration](#dataset)
3. [Image Preprocessing](#preprocessing)
4. [Feature Extraction](#features)
5. [Similarity Model](#model)
6. [Evaluation Metrics](#evaluation)
7. [Visualisation & Results](#results)
8. [Discussion & Future Work](#discussion)


## 1. Strategy & Approach <a id="strategy"></a>

### Problem Framing
Finding similar paintings is an **unsupervised / self-supervised retrieval** problem.  
Given a query painting, we want to rank the collection by visual similarity.

We break similarity into three semantic levels:

| Level | What we capture | Technique |
|-------|----------------|-----------|
| **Low-level** | Colour palette, texture, brushstroke | Colour histograms, LBP |
| **Mid-level** | Composition, shapes, edges | HOG, SIFT bag-of-words |
| **High-level** | Subject matter, faces, poses | Deep CNN features (ResNet/CLIP) |

### Chosen Pipeline (offline environment – no GPU/internet)
Because this notebook must run **without an internet connection**, we implement the classical pipeline:

```
Raw Image → Resize → Multi-feature Extraction → Normalise → PCA → Cosine Similarity Index
```

Feature vector = `[colour_hist | HOG | LBP | Hu_moments]`

When a GPU and PyTorch/CLIP are available (production setting), the deep-feature branch described in Section 8 should be preferred — it significantly outperforms hand-crafted features on high-level semantic similarity.

### Why cosine similarity?
- Scale-invariant (different image sizes produce comparable unit vectors)
- Fast $O(d)$ per pair; $O(n)$ with matrix multiplication for a full query
- Works well after L2-normalisation in high-dimensional spaces


## 2. Imports & Configuration

In [ ]:
import os
import csv
import json
import random
import warnings
import zipfile
import io
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from PIL import Image

from scipy.spatial.distance import cosine, cdist
from scipy.stats import kendalltau, spearmanr

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    pairwise_distances
)
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray, rgb2hsv
from skimage.transform import resize as sk_resize
from skimage.filters import sobel
from skimage import img_as_float

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

print("All imports successful.")
print(f"NumPy {np.__version__} | Pandas {pd.__version__} | Scikit-learn {__import__('sklearn').__version__}")


## 3. Dataset Loading & Exploration <a id="dataset"></a>

The **National Gallery of Art (NGA) Open Data** repository provides:
- `data/objects.csv` – metadata for ~140 000 artworks (title, artist, date, medium, …)
- `data/published_images.csv` – IIIF image URLs for each object
- Images accessible via the IIIF API: `https://api.nga.gov/iiif/{uuid}/full/!200,200/0/default.jpg`

Because network access is disabled in this environment, we:
1. Demonstrate the **full download pipeline** (commented out).
2. Generate a **realistic synthetic dataset** that mimics the NGA CSV schema and image properties, so every downstream cell runs end-to-end.


In [ ]:
# ── Helper: load NGA CSVs if available locally ──────────────────────────────
NGA_DATA_DIR = Path("nga_data")          # set to your local clone of the repo
OBJECTS_CSV  = NGA_DATA_DIR / "data/objects.csv"
IMAGES_CSV   = NGA_DATA_DIR / "data/published_images.csv"
IMG_CACHE    = Path("nga_images")

USE_REAL_DATA = OBJECTS_CSV.exists() and IMAGES_CSV.exists()
print(f"Real NGA data found: {USE_REAL_DATA}")


In [ ]:
# ── Real download pipeline (runs only when USE_REAL_DATA=True) ───────────────
def load_nga_metadata(max_rows: int = 5000) -> pd.DataFrame:
    """Load and merge NGA objects + published_images CSVs."""
    obj = pd.read_csv(OBJECTS_CSV, low_memory=False)
    img = pd.read_csv(IMAGES_CSV,  low_memory=False)
    merged = obj.merge(img, left_on='objectid', right_on='depictstmsobjectid', how='inner')
    # keep paintings only
    merged = merged[merged['medium'].str.contains('oil|tempera|acrylic|paint', case=False, na=False)]
    merged = merged.dropna(subset=['iiifthumburl'])
    print(f"Paintings with images: {len(merged):,}")
    return merged.head(max_rows)

def download_image(url: str, size: int = 200) -> Image.Image | None:
    """Fetch one image from the NGA IIIF endpoint."""
    import urllib.request, time
    # Replace size token in IIIF URL
    url = url.replace('/full/', f'/full/!{size},{size}/0/')
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return Image.open(io.BytesIO(r.read())).convert('RGB')
    except Exception as e:
        return None

def download_dataset(df: pd.DataFrame, n: int = 500) -> list[dict]:
    """Download n painting images; return list of {id, title, artist, image}."""
    IMG_CACHE.mkdir(exist_ok=True)
    records, sample = [], df.sample(min(n, len(df)))
    for _, row in sample.iterrows():
        fpath = IMG_CACHE / f"{row['objectid']}.jpg"
        if fpath.exists():
            img = Image.open(fpath).convert('RGB')
        else:
            img = download_image(row['iiifthumburl'])
            if img is None: continue
            img.save(fpath)
        records.append({'id': row['objectid'], 'title': row.get('title',''), 
                        'artist': row.get('attribution',''), 'image': img})
    return records

if USE_REAL_DATA:
    meta_df = load_nga_metadata(5000)
    records  = download_dataset(meta_df, n=500)
    print(f"Downloaded {len(records)} paintings")


In [ ]:
# ── Synthetic dataset (mimics NGA content when offline) ─────────────────────
def make_synthetic_image(style: str, size: int = 128) -> Image.Image:
    """Generate a plausible painting-like synthetic image."""
    rng = np.random.default_rng()
    canvas = np.zeros((size, size, 3), dtype=np.uint8)

    if style == 'portrait':
        # Warm background
        canvas[:] = rng.integers(180, 220, 3)
        # Oval face region
        cy, cx = size//3, size//2
        for y in range(size):
            for x in range(size):
                if ((x-cx)**2/(size*0.15)**2 + (y-cy)**2/(size*0.22)**2) < 1:
                    canvas[y, x] = [220+rng.integers(-20,20),
                                    190+rng.integers(-20,20),
                                    160+rng.integers(-20,20)]
        # Dark clothing lower half
        canvas[int(size*0.6):] = rng.integers(30, 80, 3)

    elif style == 'landscape':
        # Sky gradient
        for y in range(size//2):
            v = int(100 + 155*(y/(size//2)))
            canvas[y] = [v//3, v//2, v]
        # Ground
        canvas[size//2:] = [rng.integers(50,120), rng.integers(80,160), rng.integers(30,80)]
        # Random foliage blobs
        for _ in range(rng.integers(3, 8)):
            cy_ = rng.integers(size//3, 2*size//3)
            cx_ = rng.integers(10, size-10)
            r = rng.integers(8, 20)
            col = [rng.integers(20,80), rng.integers(100,200), rng.integers(20,80)]
            ys, xs = np.ogrid[:size, :size]
            mask = (xs-cx_)**2 + (ys-cy_)**2 < r**2
            canvas[mask] = col

    elif style == 'abstract':
        # Random geometric shapes with vivid colours
        canvas[:] = rng.integers(200, 255, 3)
        for _ in range(rng.integers(5, 15)):
            col = rng.integers(0, 255, 3)
            x1, y1 = rng.integers(0, size, 2)
            x2, y2 = rng.integers(0, size, 2)
            canvas[min(y1,y2):max(y1,y2), min(x1,x2):max(x1,x2)] = col

    elif style == 'religious':
        # Gold background + central figure
        canvas[:] = [218, 165, 32]
        cx_ = size//2
        canvas[size//4:, cx_-size//8:cx_+size//8] = [30, 80, 180]
        canvas[size//6:size//3, cx_-size//12:cx_+size//12] = [240, 210, 170]

    elif style == 'still_life':
        # Dark background, colourful objects
        canvas[:] = [30, 25, 20]
        for _ in range(rng.integers(3, 7)):
            cx_ = rng.integers(15, size-15)
            cy_ = rng.integers(size//2, size-15)
            r = rng.integers(10, 22)
            col = rng.integers(100, 255, 3)
            ys, xs = np.ogrid[:size, :size]
            canvas[np.where((xs-cx_)**2 + (ys-cy_)**2 < r**2)] = col

    # Add noise (simulate texture)
    noise = rng.integers(-15, 15, canvas.shape, dtype=np.int16)
    canvas = np.clip(canvas.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(canvas)


STYLES = ['portrait', 'landscape', 'abstract', 'religious', 'still_life']
STYLE_PROPS = {'portrait':0.30, 'landscape':0.25, 'abstract':0.15,
               'religious':0.15, 'still_life':0.15}
N_SYNTHETIC = 300

# Use real NGA records when available; otherwise generate synthetic fallback.
if USE_REAL_DATA and 'records' in globals() and len(records) > 0:
    for rec in records:
        rec.setdefault('style', 'unknown')
    print(f"Dataset: {len(records)} paintings (real NGA subset)")
else:
    synthetic_styles = random.choices(STYLES, weights=list(STYLE_PROPS.values()), k=N_SYNTHETIC)
    synthetic_records = []
    for i, style in enumerate(synthetic_styles):
        img = make_synthetic_image(style, size=128)
        synthetic_records.append({
            'id': f'synth_{i:04d}',
            'title': f'Synthetic {style.replace("_", " ").title()} #{i}',
            'artist': f'Artist_{i % 30:02d}',
            'style': style,
            'image': img
        })

    records = synthetic_records
    print(f"Dataset: {len(records)} paintings (synthetic fallback)")
    print("Style distribution:", Counter(r['style'] for r in records))


In [ ]:
# ── Quick dataset visualisation ──────────────────────────────────────────────
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Sample Paintings from Dataset', fontsize=16, color='white', 
             fontfamily='serif', y=1.01)

sampled = random.sample(records, 18)
for ax, rec in zip(axes.flat, sampled):
    ax.imshow(rec['image'])
    ax.set_title(rec.get('style','?'), fontsize=7, color='#aaddff', pad=2)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#334466')

plt.tight_layout()
plt.savefig('sample_paintings.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Figure saved.")


## 4. Image Preprocessing <a id="preprocessing"></a>

All images are resized to **128 × 128 px** before feature extraction.  
We keep both the **RGB** and **Lab** colour spaces:

- RGB → colour histograms, HOG  
- Lab → perceptually uniform distance (useful for colour-based similarity)  
- Greyscale → LBP texture, Hu moments  


In [ ]:
TARGET_SIZE = (128, 128)

def preprocess_image(img: Image.Image) -> dict:
    """Return a dict of resized arrays in multiple colour spaces."""
    img_rgb = img.convert('RGB').resize(TARGET_SIZE, Image.LANCZOS)
    arr     = np.array(img_rgb, dtype=np.float32) / 255.0
    arr_u8  = (arr * 255).astype(np.uint8)
    
    # Greyscale
    grey   = rgb2gray(arr)
    # HSV
    hsv    = rgb2hsv(arr)
    
    return {'rgb': arr, 'rgb_u8': arr_u8, 'grey': grey, 'hsv': hsv}

# Pre-process all records
print("Preprocessing images...")
for rec in records:
    rec['proc'] = preprocess_image(rec['image'])
print("Done.")


## 5. Feature Extraction <a id="features"></a>

We extract four complementary feature groups and concatenate them into one vector:

| Feature | Dims | Captures |
|---------|------|---------|
| RGB colour histogram (32 bins × 3) | 96 | Colour palette |
| HSV histogram (16 bins × 3) | 48 | Hue/saturation distribution |
| HOG | 324 | Edges, gradients, shapes |
| LBP (radius=2, points=16) | 18 | Texture patterns |
| Hu moments | 7 | Global shape/composition |
| **Total** | **493** | |

All features are **L2-normalised** individually before concatenation; the combined vector is also L2-normalised.


In [ ]:
def extract_colour_hist(proc: dict, bins: int = 32) -> np.ndarray:
    """Per-channel RGB + HSV histograms, concatenated."""
    rgb_feats, hsv_feats = [], []
    for ch in range(3):
        h, _ = np.histogram(proc['rgb'][:,:,ch], bins=bins, range=(0,1))
        rgb_feats.append(h.astype(float))
    for ch in range(3):
        h, _ = np.histogram(proc['hsv'][:,:,ch], bins=bins//2, range=(0,1))
        hsv_feats.append(h.astype(float))
    return np.concatenate(rgb_feats + hsv_feats)

def extract_hog_features(proc: dict) -> np.ndarray:
    """HOG descriptor on greyscale image."""
    feat = hog(proc['grey'],
               orientations=9,
               pixels_per_cell=(16, 16),
               cells_per_block=(2, 2),
               block_norm='L2-Hys',
               feature_vector=True)
    return feat

def extract_lbp_features(proc: dict, radius: int = 2, n_points: int = 16) -> np.ndarray:
    """Local Binary Pattern histogram (uniform)."""
    lbp = local_binary_pattern(proc['grey'], n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_points+2,
                           range=(0, n_points+2), density=True)
    return hist.astype(float)

def extract_hu_moments(proc: dict) -> np.ndarray:
    """Hu moments from edge magnitude image."""
    from skimage.measure import moments, moments_hu
    edges = sobel(proc['grey'])
    m = moments(edges, order=3)
    hu = moments_hu(m)
    # Log-transform for numerical stability
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

def l2(v: np.ndarray) -> np.ndarray:
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else v

def build_feature_vector(proc: dict) -> np.ndarray:
    """Build full feature vector for one painting."""
    colour = l2(extract_colour_hist(proc))
    hog_f  = l2(extract_hog_features(proc))
    lbp    = l2(extract_lbp_features(proc))
    hu     = l2(extract_hu_moments(proc))
    full   = np.concatenate([colour, hog_f, lbp, hu])
    return l2(full)

# Extract features for all paintings
print("Extracting features ...")
feat_matrix = []
for i, rec in enumerate(records):
    fv = build_feature_vector(rec['proc'])
    feat_matrix.append(fv)
    rec['feature_vector'] = fv
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(records)}")

feat_matrix = np.array(feat_matrix, dtype=np.float32)
print(f"\nFeature matrix shape: {feat_matrix.shape}")
print(f"Feature stats — mean: {feat_matrix.mean():.4f}, std: {feat_matrix.std():.4f}")


In [ ]:
# ── PCA dimensionality reduction ─────────────────────────────────────────────
pca = PCA(n_components=0.95, random_state=42)   # keep 95% variance
feat_pca = pca.fit_transform(feat_matrix)

print(f"PCA: {feat_matrix.shape[1]} → {feat_pca.shape[1]} dimensions "
      f"({pca.explained_variance_ratio_.sum()*100:.1f}% variance retained)")

# Re-normalise after PCA
feat_pca = normalize(feat_pca, norm='l2')

# Explained variance plot
fig, ax = plt.subplots(figsize=(8, 3.5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(cumvar, color='#58a6ff', linewidth=2)
ax.axhline(95, color='#f85149', linestyle='--', linewidth=1, label='95% threshold')
ax.fill_between(range(len(cumvar)), cumvar, alpha=0.15, color='#58a6ff')
ax.set_xlabel('Number of Components', color='#c9d1d9')
ax.set_ylabel('Cumulative Explained Variance (%)', color='#c9d1d9')
ax.set_title('PCA Explained Variance', color='#f0f6fc', fontsize=13, pad=10)
ax.legend(facecolor='#21262d', labelcolor='#c9d1d9')
ax.tick_params(colors='#8b949e')
for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


## 6. Similarity Model <a id="model"></a>

### 6.1 Cosine Similarity Index
We build a **brute-force cosine nearest-neighbour index** from the PCA-reduced features.  
For large collections a proper ANN index (FAISS, Annoy, ScaNN) should replace this.


In [ ]:
class PaintingSimilarityIndex:
    """
    Cosine-similarity retrieval index for paintings.
    
    Parameters
    ----------
    features : np.ndarray, shape (N, D)
        L2-normalised feature matrix.
    metadata : list[dict]
        One dict per painting with at least 'id', 'title', 'style'.
    """
    def __init__(self, features: np.ndarray, metadata: list):
        self.features = features                      # already L2-normed
        self.metadata = metadata
        self.n        = len(metadata)
        print(f"Index built: {self.n} paintings, {features.shape[1]}-dim features")

    def query(self, idx: int, top_k: int = 10, exclude_self: bool = True
              ) -> list[tuple[int, float]]:
        """Return top_k most similar paintings to query at position idx."""
        qvec = self.features[idx]
        # Cosine similarity = dot product (vectors are L2-normalised)
        sims = self.features @ qvec          # shape (N,)
        ranked = np.argsort(-sims)
        results = []
        for r in ranked:
            if exclude_self and r == idx:
                continue
            results.append((int(r), float(sims[r])))
            if len(results) >= top_k:
                break
        return results

    def similarity_matrix(self) -> np.ndarray:
        """Full N×N cosine similarity matrix."""
        return self.features @ self.features.T


index = PaintingSimilarityIndex(feat_pca, records)


In [ ]:
# ── Demo query ───────────────────────────────────────────────────────────────
QUERY_IDX = 0   # change to any painting index

query_rec = records[QUERY_IDX]
top10 = index.query(QUERY_IDX, top_k=10)

print(f"Query: [{query_rec['id']}] '{query_rec['title']}' ({query_rec.get('style','?')})")
print(f"{'Rank':>4}  {'ID':>12}  {'Style':>12}  {'Similarity':>10}")
print("-" * 48)
for rank, (ridx, sim) in enumerate(top10, 1):
    rec = records[ridx]
    print(f"{rank:>4}  {rec['id']:>12}  {rec.get('style','?'):>12}  {sim:>10.4f}")


In [ ]:
# ── Visual query result grid ──────────────────────────────────────────────────
def plot_query_results(query_idx: int, top_k: int = 8, title_prefix: str = ""):
    results   = index.query(query_idx, top_k=top_k)
    query_rec = records[query_idx]
    ncols     = top_k // 2
    fig, axes = plt.subplots(2, ncols + 1, figsize=(3*(ncols+1), 7))
    fig.patch.set_facecolor('#0d1117')
    
    # Query image (left column, both rows merged conceptually)
    for row in range(2):
        ax = axes[row, 0]
        if row == 0:
            ax.imshow(query_rec['image'])
            ax.set_title(f"QUERY\n{query_rec.get('style','?')}",
                         color='#ffa657', fontsize=9, fontweight='bold')
        else:
            ax.axis('off')
            ax.text(0.5, 0.5, 'Query', ha='center', va='center',
                    color='#ffa657', fontsize=11)
        ax.axis('off')
    
    # Results
    for i, (ridx, sim) in enumerate(results):
        row, col = divmod(i, ncols)
        if row >= 2 or col+1 >= ncols+1: break
        ax = axes[row, col+1]
        ax.imshow(records[ridx]['image'])
        style = records[ridx].get('style','?')
        match = '✓' if style == query_rec.get('style') else ' '
        ax.set_title(f"{match} {style}\nsim={sim:.3f}",
                     color='#58a6ff', fontsize=8)
        ax.axis('off')
    
    q_style = query_rec.get('style','?')
    fig.suptitle(f"{title_prefix}Top-{top_k} Similar Paintings  |  Query style: {q_style}",
                 color='#f0f6fc', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(f'query_result_{query_idx}.png', dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()

plot_query_results(QUERY_IDX, top_k=8)
# Try a few more queries
for qi in [15, 42, 100]:
    if qi < len(records):
        plot_query_results(qi, top_k=8)


## 7. Evaluation Metrics <a id="evaluation"></a>

### 7.1 Supervised Metrics (when ground-truth labels are available)

We treat **painting style/genre** as a proxy ground-truth label.  
For each query we retrieve the top-K results and compute:

| Metric | Formula | Why |
|--------|---------|-----|
| **Precision@K** | $\frac{\|\text{relevant in top-K}\|}{K}$ | How many results are truly similar |
| **Recall@K** | $\frac{\|\text{relevant in top-K}\|}{\|\text{all relevant}\|}$ | Coverage |
| **Mean Average Precision (mAP)** | $\frac{1}{Q}\sum_q AP_q$ | Rank-aware precision |
| **NDCG@K** | Normalised discounted cumulative gain | Rewards high-similarity items at top |

### 7.2 Unsupervised Metrics

When no labels exist we assess the **quality of the feature space** using clustering:

| Metric | Range | Better |
|--------|-------|--------|
| **Silhouette Score** | [-1, 1] | Higher |
| **Davies-Bouldin Index** | [0, ∞) | Lower |
| **Calinski-Harabász** | [0, ∞) | Higher |

### 7.3 Consistency Metrics

- **Intra-class distance** (mean pairwise distance within same style) – should be lower
- **Inter-class distance** (mean pairwise distance across styles) – should be higher
- **Separation ratio** = inter / intra > 1 is desirable


In [ ]:
# ── Precision@K, Recall@K, AP for each query ─────────────────────────────────
styles_arr = np.array([r.get('style','?') for r in records])
K_VALUES   = [5, 10, 20]

def average_precision(query_idx: int, k: int) -> float:
    q_style  = styles_arr[query_idx]
    results  = index.query(query_idx, top_k=k)
    hits, ap = 0, 0.0
    for rank, (ridx, _) in enumerate(results, 1):
        if styles_arr[ridx] == q_style:
            hits += 1
            ap   += hits / rank
    n_relevant = max(1, np.sum(styles_arr == q_style) - 1)  # exclude self
    return ap / min(k, n_relevant)

def precision_at_k(query_idx: int, k: int) -> float:
    q_style = styles_arr[query_idx]
    results = index.query(query_idx, top_k=k)
    return sum(1 for ridx, _ in results if styles_arr[ridx] == q_style) / k

def recall_at_k(query_idx: int, k: int) -> float:
    q_style    = styles_arr[query_idx]
    n_relevant = max(1, np.sum(styles_arr == q_style) - 1)
    results    = index.query(query_idx, top_k=k)
    hits       = sum(1 for ridx, _ in results if styles_arr[ridx] == q_style)
    return hits / n_relevant

# Evaluate over all queries
sample_queries = list(range(len(records)))   # full evaluation
metric_rows = []
for qi in sample_queries:
    row = {'id': records[qi]['id'], 'style': styles_arr[qi]}
    for k in K_VALUES:
        row[f'P@{k}']  = precision_at_k(qi, k)
        row[f'R@{k}']  = recall_at_k(qi, k)
        row[f'AP@{k}'] = average_precision(qi, k)
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
summary    = metrics_df.drop(columns=['id','style']).mean()

print("=" * 55)
print("   RETRIEVAL EVALUATION SUMMARY (mean over all queries)")
print("=" * 55)
for k in K_VALUES:
    print(f"\n  K = {k}")
    print(f"    Precision@{k}  : {summary[f'P@{k}']:.4f}")
    print(f"    Recall@{k}     : {summary[f'R@{k}']:.4f}")
    print(f"    mAP@{k}        : {summary[f'AP@{k}']:.4f}")


In [ ]:
# ── Per-style breakdown ───────────────────────────────────────────────────────
style_metrics = metrics_df.groupby('style')[[f'P@{k}' for k in K_VALUES] + 
                                             [f'AP@{k}' for k in K_VALUES]].mean()
print("\nPer-style Precision@K:")
print(style_metrics[[f'P@{k}' for k in K_VALUES]].round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#0d1117')
colours = ['#58a6ff', '#3fb950', '#f85149']

for ax, k, col in zip(axes, K_VALUES, colours):
    ax.set_facecolor('#161b22')
    vals   = style_metrics[f'P@{k}'].sort_values(ascending=False)
    bars   = ax.bar(vals.index, vals.values, color=col, alpha=0.85, edgecolor='#30363d')
    ax.axhline(summary[f'P@{k}'], color='white', linestyle='--', linewidth=1,
               label=f'Mean={summary[f"P@{k}"]:.3f}')
    ax.set_title(f'Precision @ {k}', color='#f0f6fc', fontsize=11)
    ax.set_ylabel('P@K', color='#c9d1d9')
    ax.tick_params(colors='#8b949e', axis='both', labelrotation=30)
    ax.legend(facecolor='#21262d', labelcolor='#c9d1d9', fontsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

fig.suptitle('Precision@K by Painting Style', color='#f0f6fc', fontsize=13)
plt.tight_layout()
plt.savefig('precision_by_style.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# ── Clustering evaluation (unsupervised) ─────────────────────────────────────
style_labels      = pd.Categorical(styles_arr).codes
n_clusters_true   = len(set(styles_arr))

print("Clustering quality of feature space:")
print(f"  True number of styles: {n_clusters_true}")
print()

# K-Means with true k
km = KMeans(n_clusters=n_clusters_true, random_state=42, n_init=10)
km_labels = km.fit_predict(feat_pca)

sil  = silhouette_score(feat_pca, km_labels, metric='cosine')
db   = davies_bouldin_score(feat_pca, km_labels)
ch   = calinski_harabasz_score(feat_pca, km_labels)

print(f"  K-Means (k={n_clusters_true}) vs feature space:")
print(f"    Silhouette Score (cosine)  : {sil:.4f}  (1.0 = perfect)")
print(f"    Davies-Bouldin Index       : {db:.4f}  (0.0 = perfect)")
print(f"    Calinski-Harabász Score    : {ch:.2f}  (higher = better)")

# Also check against ground-truth labels
sil_gt = silhouette_score(feat_pca, style_labels, metric='cosine')
print(f"\n  Ground-truth style labels:")
print(f"    Silhouette Score (cosine)  : {sil_gt:.4f}")


In [ ]:
# ── Intra vs. Inter class distance ───────────────────────────────────────────
print("\nIntra-class vs Inter-class cosine distances:")
print(f"  {'Style':>14}  {'Intra-dist':>10}  {'Samples':>7}")
print("  " + "-"*36)

intra_all, inter_pairs = [], []
unique_styles = list(set(styles_arr))
style_feats   = {s: feat_pca[styles_arr == s] for s in unique_styles}

for style in unique_styles:
    sf = style_feats[style]
    if len(sf) < 2: continue
    d  = pairwise_distances(sf, metric='cosine')
    iu = np.triu_indices(len(sf), k=1)
    intra_all.extend(d[iu].tolist())
    print(f"  {style:>14}  {d[iu].mean():>10.4f}  {len(sf):>7}")

# Inter-class: sample across styles
for i, s1 in enumerate(unique_styles):
    for s2 in unique_styles[i+1:]:
        d_cross = pairwise_distances(style_feats[s1], style_feats[s2], metric='cosine')
        inter_pairs.extend(d_cross.ravel().tolist())

mean_intra = np.mean(intra_all)
mean_inter = np.mean(inter_pairs)
sep_ratio  = mean_inter / mean_intra if mean_intra > 0 else float('inf')

print(f"\n  Mean intra-class distance : {mean_intra:.4f}")
print(f"  Mean inter-class distance : {mean_inter:.4f}")
print(f"  Separation ratio          : {sep_ratio:.4f}  (>1 is desirable)")


## 8. Visualisation & Results <a id="results"></a>

In [ ]:
# ── t-SNE embedding of the feature space ────────────────────────────────────
print("Computing t-SNE (this may take a minute) ...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42,
            metric='cosine', init='random', learning_rate='auto')
emb  = tsne.fit_transform(feat_pca)

STYLE_COLOURS = {
    'portrait':   '#f85149',
    'landscape':  '#3fb950',
    'abstract':   '#d2a8ff',
    'religious':  '#ffa657',
    'still_life': '#58a6ff',
}

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

for style, col in STYLE_COLOURS.items():
    mask = styles_arr == style
    ax.scatter(emb[mask, 0], emb[mask, 1],
               c=col, s=22, alpha=0.75, label=style.replace('_',' ').title(),
               linewidths=0)

ax.set_title('t-SNE of Painting Feature Space', color='#f0f6fc',
             fontsize=14, pad=12, fontfamily='serif')
ax.set_xlabel('t-SNE 1', color='#8b949e')
ax.set_ylabel('t-SNE 2', color='#8b949e')
ax.tick_params(colors='#8b949e')
for sp in ax.spines.values(): sp.set_edgecolor('#21262d')

leg = ax.legend(facecolor='#161b22', labelcolor='#c9d1d9',
                framealpha=0.9, fontsize=10, title='Style',
                title_fontsize=9)
leg.get_title().set_color('#c9d1d9')

plt.tight_layout()
plt.savefig('tsne_embedding.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# ── Similarity matrix heatmap (subset) ───────────────────────────────────────
N_SHOW  = 60
indices = []
for style in STYLES:
    mask = np.where(styles_arr == style)[0]
    indices.extend(mask[:N_SHOW//len(STYLES)].tolist())
indices = sorted(indices)[:N_SHOW]

sub_feats = feat_pca[indices]
sub_sims  = sub_feats @ sub_feats.T
sub_styles= styles_arr[indices]

fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('#0d1117')
im = ax.imshow(sub_sims, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title(f'Similarity Matrix (N={N_SHOW}, sorted by style)',
             color='#f0f6fc', fontsize=12)
ax.set_xlabel('Painting Index', color='#c9d1d9')
ax.set_ylabel('Painting Index', color='#c9d1d9')
ax.tick_params(colors='#8b949e')
for sp in ax.spines.values(): sp.set_edgecolor('#21262d')

# Draw style boundaries
style_sizes = [np.sum(sub_styles == s) for s in STYLES if np.sum(sub_styles==s)>0]
boundaries  = np.cumsum(style_sizes)[:-1] - 0.5
for b in boundaries:
    ax.axhline(b, color='white', linewidth=0.8, alpha=0.6)
    ax.axvline(b, color='white', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig('similarity_matrix.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


In [ ]:
# ── Precision-Recall curve ───────────────────────────────────────────────────
k_range = list(range(1, min(51, len(records))))
mean_p  = []
mean_r  = []
for k in k_range:
    ps = [precision_at_k(qi, k) for qi in range(len(records))]
    rs = [recall_at_k(qi, k)    for qi in range(len(records))]
    mean_p.append(np.mean(ps))
    mean_r.append(np.mean(rs))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0d1117')

for ax in axes:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')
    ax.tick_params(colors='#8b949e')

axes[0].plot(k_range, mean_p, color='#58a6ff', linewidth=2)
axes[0].fill_between(k_range, mean_p, alpha=0.15, color='#58a6ff')
axes[0].set_title('Mean Precision@K', color='#f0f6fc')
axes[0].set_xlabel('K', color='#c9d1d9')
axes[0].set_ylabel('Precision', color='#c9d1d9')

axes[1].plot(mean_r, mean_p, color='#3fb950', linewidth=2, marker='.')
axes[1].set_title('Precision–Recall Curve', color='#f0f6fc')
axes[1].set_xlabel('Mean Recall', color='#c9d1d9')
axes[1].set_ylabel('Mean Precision', color='#c9d1d9')

fig.suptitle('Retrieval Performance', color='#f0f6fc', fontsize=13)
plt.tight_layout()
plt.savefig('pr_curve.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


## 9. Discussion & Future Work <a id="discussion"></a>

### Current Pipeline Summary

| Component | Choice | Justification |
|-----------|--------|---------------|
| Feature extraction | HOG + Colour histogram + LBP + Hu moments | Interpretable, no GPU needed, fast |
| Dimensionality reduction | PCA (95% variance) | Removes noise, speeds up search |
| Similarity metric | Cosine similarity | Scale-invariant, works with L2-normed vectors |
| Index | Brute-force matrix multiply | Sufficient for ≤10k paintings; O(N·D) |

### Limitations of Hand-Crafted Features
1. **Semantic gap**: A portrait of a man and a portrait of a woman have similar HOG/colour features but may differ in who is depicted.
2. **Pose variability**: HOG can struggle with large rotations or scale changes.
3. **Style nuance**: Distinguishing Baroque from Renaissance portraiture requires higher-level understanding.

### Recommended Production Pipeline

```
Image
  └─► CLIP ViT-L/14 (or ResNet-50 trained on NGA labels)
         ↓
      512-dim embedding (already L2-normed)
         ↓
      FAISS IVF index (cosine, ~1ms per query for 100k paintings)
         ↓
      Top-K results
```

**Why CLIP?**
- Zero-shot – no labelled painting data needed
- Encodes both **visual content** and **semantic concepts**
- Multi-modal: supports text queries like *"portrait with blue background"*

### Evaluation Improvements
- Collect **human relevance judgements** via a pairwise comparison interface (AMT / Label Studio)
- Use **NDCG** over binary precision once graded relevance is available
- Cross-validate with art-historian labels from the NGA `classification` fields (school, period, style)

### Scaling
| Scale | Recommended index |
|-------|------------------|
| < 50k | Brute-force cosine (numpy) |
| 50k–5M | FAISS IVF-Flat or HNSW |
| > 5M | FAISS IVF-PQ with OPQ pre-processing |

### Next Steps for ArtExtract
1. Fine-tune a CLIP or ResNet model on NGA labels for domain adaptation
2. Add **face detection** (MTCNN) + **pose estimation** (ViTPose) as auxiliary feature branches for portrait similarity
3. Add multimodal retrieval (image + text) for richer similarity search


In [ ]:
# ── Final summary table ──────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════╗")
print("║           ARTEXTRACT TASK 2 – FINAL SUMMARY              ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Dataset size          : {len(records):>5} paintings                   ║")
print(f"║  Feature dimensionality: {feat_matrix.shape[1]:>5} (raw)                     ║")
print(f"║  After PCA             : {feat_pca.shape[1]:>5} dims (95% var)              ║")
print(f"║  Similarity metric     : Cosine                          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  mAP@5                 : {summary['AP@5']:>6.4f}                          ║")
print(f"║  mAP@10                : {summary['AP@10']:>6.4f}                          ║")
print(f"║  Precision@10          : {summary['P@10']:>6.4f}                          ║")
print(f"║  Silhouette (cosine)   : {sil:>6.4f}                          ║")
print(f"║  Separation ratio      : {sep_ratio:>6.4f}                          ║")
print("╚══════════════════════════════════════════════════════════╝")
